# 04 — Modeling & Evaluation
**Deliverable D | Qiyas AI Hackathon | Team Quatro**

This notebook executes the modeling and evaluation workflow for tasks D1 through D9:
- **D1. Baselines** (Mean predictor, Seasonal-Naive)
- **D2. Model Comparison** (Ridge, RF, HistGB, LightGBM)
- **D3. Rolling-Origin Validation** (4 × 14-day expanding folds)
- **D4. Feature Availability & Leakage Audit**
- **D5. Ablation Study** (Calendar $	o$ Weather $	o$ Events $	o$ Both)
- **D6. Hyperparameter Tuning** (TimeSeriesSplit)
- **D7. Error Analysis** (Zone, hour, day type, and extreme error zone-hours)
- **D8. Response to Findings**
- **D9. Operational Plain-Language Metric**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

from train import run_pipeline, CONFIG

## Execute the Complete D1–D9 Pipeline
We run the modular modeling pipeline on the chronological October holdout split:

In [ ]:
results = run_pipeline(CONFIG)
print("Pipeline execution complete!")

## D1. Baselines Performance
Seasonal-naive beats the global mean predictor by ~2x on RMSE (17.16 vs 30.16) and ~3x on MAE (7.53 vs 20.79).

In [ ]:
display(results['baselines'])

## D2. Model Family Comparison
Evaluating multiple model families on the exact same chronological split and feature set:

In [ ]:
display(results['comparison'])
print("LightGBM selected: Ties Random Forest on accuracy (15.79 vs 15.73) with 10x faster training time and native categorical zone handling.")

## D3. Rolling-Origin Cross-Validation (4 Folds × 14 Days)

In [ ]:
display(results['rolling'])
lgbm_scores = results['rolling']['lgbm_rmse']
print(f"Rolling-Origin RMSE: {lgbm_scores.mean():.3f} +/- {lgbm_scores.std():.3f}")

## D4. Leakage Audit
Proving the artificial, non-reproducible gain of leaky operational features:

In [ ]:
leakage_table = pd.DataFrame([
    {"Model": "Clean Baseline (Calendar+Trend)", "RMSE": 16.92, "Status": "Legitimate (Forecast-Time Known)"},
    {"Model": "Leaky Model (+ active_drivers, wait, fare)", "RMSE": 13.55, "Status": "TARGET LEAKAGE (19.9% Artificial Gain)"},
    {"Model": "Full Clean Model (+ Weather + Events)", "RMSE": 15.79, "Status": "Legitimate External Signal (Rule 5)"}
])
display(leakage_table)

## D5. Ablation Study
Quantifying the incremental value of each multi-table join on identical splits:

In [ ]:
display(results['ablation'])
print("Takeaway: Weather alone improves RMSE by 0.85; events add 0.20; both together drop RMSE by 1.14.")

## D6. Hyperparameter Tuning
TimeSeriesSplit randomized search parameters and score improvements:

In [ ]:
tuning_table = pd.DataFrame([
    {"Model": "Default LightGBM", "RMSE": 15.785, "MAE": 6.764},
    {"Model": "Tuned LightGBM", "RMSE": 15.732, "MAE": 6.751}
])
display(tuning_table)
print(f"Best Tuned Parameters: {results['tuning']['best_params']}")

## D7. Error Analysis
Inspecting error distributions across zones, hours, and extreme spike hypotheses:

In [ ]:
print("Top 10 Largest-Error Zone-Hours:")
display(results['errors']['top_errors'])

## D8. Response to Findings
Testing `is_payday` feature + prediction clipping (honest reporting of findings):

In [ ]:
display(results['response'])

## D9. Operational Plain-Language Metric
Translating model error into fleet planning reality:

In [ ]:
ops = results['plain_metric']
print(f"Mean City Demand           : {ops['mean_trips_per_zone_hour']} trips / zone-hour")
print(f"Final Model RMSE           : {ops['rmse_trips']} trips ({ops['rmse_pct_of_mean']}% of mean)")
print(f"Final Model MAE            : {ops['mae_in_drivers']} drivers absolute deviation")
print(f"Peak Buffer Planning Error : {ops['rmse_in_drivers']} drivers surge capacity buffer")